# 🔱 VoiceBatch Studio v2.2.5 - [Strict Hindi Lock]
जापानी/चाइनीज भाषा के मिश्रण को रोकने के लिए सख्त फिल्टर लगाया गया है।

In [ ]:
# @title 💤 Step 1: इंस्टॉलेशन
import os
!pip install -q gradio librosa soundfile coqui-tts torchcodec
os.makedirs("outputs", exist_ok=True)
print("✅ इंजन तैयार है!")

In [ ]:
# @title 🚀 Step 2: app.py (Language Enforcement & Download Fix)
app_code = r'''
import gradio as gr
import torch
from TTS.api import TTS
import librosa, soundfile as sf
import re, os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)

def strict_hindi_filter(text):
    # केवल शुद्ध देवनागरी (अ-ह), मात्राएँ, पूर्ण विराम और नंबर्स को अनुमति दें
    # यह किसी भी जापानी/चीनी फोनेम्स को आने से रोकेगा
    allowed_pattern = re.compile(r'[^\u0900-\u097F\s।,?!:;0-9]')
    clean_text = allowed_pattern.sub('', text)
    return clean_text

def studio_pro_engine(text, audio_sample, speed, pitch, lang, sil_rem):
    if not audio_sample: return None
    
    # अगर हिंदी चुनी है, तो फिल्टर लगाएं
    if lang == 'hi':
        text = strict_hindi_filter(text)
    
    out_filename = 'VoiceBatch_Studio_Output.wav'
    out_path = os.path.join('outputs', out_filename)
    
    # Generation
    tts.tts_to_file(
        text=text, 
        speaker_wav=audio_sample, 
        language=lang, 
        file_path=out_path,
        split_sentences=True
    )
    
    y, sr = librosa.load(out_path)
    if sil_rem: y, _ = librosa.effects.trim(y, top_db=25)
    if speed != 1.0: y = librosa.effects.time_stretch(y, rate=speed)
    if pitch != 0: y = librosa.effects.pitch_shift(y, sr=sr, n_steps=pitch)
    
    sf.write(out_path, y, sr)
    return out_path

with gr.Blocks(theme=gr.themes.Soft(primary_hue="orange")) as demo:
    gr.Markdown('# 🔱 VoiceBatch Studio v2.2.5')
    with gr.Row():
        with gr.Column():
            txt = gr.Textbox(label='Hindi Script', placeholder='यहाँ केवल हिंदी में लिखें...', lines=8)
            smp = gr.Audio(label='Voice Sample', type='filepath')
            lng = gr.Dropdown(choices=['hi', 'en', 'mr'], label='Language', value='hi')
            with gr.Row():
                spd = gr.Slider(0.7, 1.4, 1.0, step=0.01, label="Speed")
                ptc = gr.Slider(-4, 4, 0, step=1, label="Pitch")
            sil = gr.Checkbox(label="Silence Remover", value=True)
            btn = gr.Button('Generate Pure Hindi Audio ⚡', variant='primary')
        with gr.Column():
            out = gr.Audio(label='Download: VoiceBatch_Studio_Output.wav')

    btn.click(studio_pro_engine, [txt, smp, spd, ptc, lng, sil], out)

demo.launch(share=True, debug=True)
'''
with open('app.py', 'w') as f: f.write(app_code)
!python app.py